# Chapter 11: Grover Algorithms

---

**Prerequisites:**
- See `Chapter02_QuantumSoftware.ipynb` for installation instructions


In [ ]:
# Setup and imports
import numpy as np
import math
from qiskit import QuantumCircuit,transpile
from IPython.display import display
from qiskit import ClassicalRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import grover_operator


from Chapter03_EngineeringOptimization_functions import (truss2x2,truss3x2,truss2x3,truss3x3,truss_grid,truss_10bar)
from Chapter08_QuantumGates_functions import (simulate_statevector,  simulate_measurements, 
                                              estimateCircuitFidelity, estimateCircuitGates,findActualHardwareRequirements)



: 

## Phase Kickback Barebones

In [ ]:
circuit = QuantumCircuit(2)  
circuit.x(0)
circuit.h(0)
circuit.barrier()
circuit.cx(1,0)
circuit.barrier()
circuit.h(0)
circuit.x(0)
display(circuit.draw('mpl'))

## Phase Kickback Example

In [ ]:
circuit = QuantumCircuit(2)  
circuit.x(0)
circuit.h(0)
circuit.ry(theta = np.pi/3, qubit=1) 
circuit.barrier()
circuit.cx(1,0)
circuit.barrier()
circuit.h(0)
circuit.x(0)
display(circuit.draw('mpl'))
psi = Statevector(circuit)
display(psi.draw('latex'))

## Phase Oracle Barebones

In [ ]:
# 4 Qubits: q0 is Ancilla, q1-q3 are Input
circuit = QuantumCircuit(4)

# 1. Prepare Ancilla (q0) in the |-> state
circuit.x(0)
circuit.h(0)
circuit.barrier()

# 2. Phase Oracle for |101> 
# Input register is (q3, q2, q1). For |101>, q2 must be flipped.
circuit.mcx([1, 2, 3], 0, ctrl_state = '101')   # Controls: q1,q2,q3. Target: q0
circuit.barrier()

# 3. Clean up Ancilla to return it to |0>
circuit.h(0)
circuit.x(0)

display(circuit.draw('mpl'))

# Verify the result
# The statevector will show a negative sign on the |1010> state 
# (where the last 0 is the ancilla q0)
psi = Statevector(circuit)
display(psi.draw('latex'))

## Phase oracle example

In [ ]:
# 4 Qubits: q0 is Ancilla, q1-q3 are Input
circuit = QuantumCircuit(4)

# 1. Prepare Ancilla (q0) in the |-> state
circuit.x(0)
circuit.h(0)

for qubit in [1, 2, 3]:
    circuit.h(qubit)
circuit.barrier()
# 2. Phase Oracle for |101> 
# Input register is (q3, q2, q1). For |101>, q2 must be flipped.
circuit.mcx([1, 2, 3], 0, ctrl_state = '101')   # Controls: q1,q2,q3. Target: q0
circuit.barrier()

# 3. Clean up Ancilla to return it to |0>
circuit.h(0)
circuit.x(0)
display(circuit.draw('mpl'))

# Verify the result
# The statevector will show a negative sign on the |1010> state 
# (where the last 0 is the ancilla q0)
psi = Statevector(circuit)
display(psi.draw('latex'))

## Qiskit's Built-in PhaseOracleGate

In [ ]:
from qiskit.circuit.library import PhaseOracleGate
from Chapter12_GroverAlgorithm_functions import ensure_variable_order

expression = 'x2 & ~x1 & ~x0' # state 100
qiskit_expr = ensure_variable_order(expression, 3)

oracle_gate = PhaseOracleGate(qiskit_expr,label='Oracle')

# Now just add it to your main circuit
n = 3 # Don't include ancilla
circuit = QuantumCircuit(n)
for qubit in range(n):
    circuit.h(qubit)
circuit.append(oracle_gate, range(n))
display(circuit.draw('mpl'))
psi = Statevector(circuit)
display(psi.draw('latex'))

In [ ]:
transpiled_circuit = transpile(circuit, basis_gates=['u3', 'cx'])
display(transpiled_circuit.draw('mpl'))

### Boolean expressions

In [ ]:
expression = '(x2 & ~x1 & ~x0) | (x2 & ~x1 & x0)' # states 100 and 101
expression = '(x2 | x0)' # 
qiskit_expr = ensure_variable_order(expression, 3)

oracle_gate = PhaseOracleGate(qiskit_expr,label='Oracle')

# Now just add it to your main circuit
n = 3 # Don't include ancilla
circuit = QuantumCircuit(n)
for qubit in range(n):
    circuit.h(qubit)
circuit.append(oracle_gate, range(n))
display(circuit.draw('mpl'))
psi = Statevector(circuit)
display(psi.draw('latex'))

### Transpiled circuit

In [ ]:
transpiled_circuit = transpile(circuit, basis_gates=['u3', 'cx'])
display(transpiled_circuit.draw('mpl'))

## Feasible trusses

In [ ]:
n_members = 7
feasible_expr = "(x0 | x1 | x2) & (x0 | x3 | x4)"
# Ensure string variables are in the correct order for the oracle
qiskit_expr = ensure_variable_order(feasible_expr, n_members)
print('Expression: {}'.format(qiskit_expr))

oracle_gate = PhaseOracleGate(qiskit_expr,label="Feasible")
circuit = QuantumCircuit(n_members)
for qubit in range(n_members):
    circuit.h(qubit)
circuit.append(oracle_gate,range(n_members))
display(circuit.draw('mpl'))
psi = Statevector(circuit)


## Single feasible truss design problem
## Classical solution

In [ ]:
from Chapter12_GroverAlgorithm_functions import get_feasible_expression

n_members = 7
# generate a Boolean expression with one feasible solution
feasible_expr = get_feasible_expression() 
count = 0
for i in range(2**n_members):
    x0, x1, x2, x3, x4, x5, x6 = [int(b) for b in format(i, '07b')]
    if eval(feasible_expr):
        count += 1
        members = [j for j, b in enumerate([x0,x1,x2,x3,x4,x5,x6]) if b]
        bitstring = format(i,'07b')[::-1]  # reverse to match Qiskit ket order
        print(f"Solution: x0..x6 = {bitstring} -> active members {members}")

print(f"Total solutions: {count}")

## Grover algorithm

In [ ]:
qiskit_expr = ensure_variable_order(feasible_expr, n_members)

oracle = PhaseOracleGate(qiskit_expr,label="Feasible")
grover_op = grover_operator(oracle)

n_members  = 7
N = 2**n_members
K = math.floor(math.pi / (4 * math.asin(math.sqrt(1 / N))))
print("Optimal number of Grover iterations:", K)

qc = QuantumCircuit(grover_op.num_qubits)
qc.h(range(n_members))  # only superpose the search qubits, not ancillas
qc.compose(grover_op.power(K), inplace=True)

cr = ClassicalRegister(n_members)
qc.add_register(cr)
for i in range(n_members):
    qc.measure(i, cr[i])
counts = simulate_measurements(qc, shots=1000)
print(counts)
display(qc.draw('mpl'))

## Grover circuit Analysis

### High level depth

In [ ]:
print("width:", qc.width()) # includes ancillas
print("depth:", qc.depth()) # high level functions only

### Estimate circuit width and depth

In [ ]:
info = estimateCircuitGates(qc)
print("Grover Circuit:", info)

## Void Detection

### Example

In [ ]:
from Chapter03_EngineeringOptimization_functions import MicrostructureGenerator

microstructure = MicrostructureGenerator(nx=4, ny=4, inclusion_fraction=0.05, micro_type='random_elements')
microstructure.plot()

### string from microstructure

In [ ]:
from Chapter12_GroverAlgorithm_functions import get_void_expression, decode_void_measurement


# ── Cell 2: Coordinate encoding ───────────────────────────────────────────────
expression, n_qubits, void_coords = get_void_expression(grid=microstructure.data)
 
grid_array = np.array(microstructure.data)
rows, cols = grid_array.shape
m = int(np.log2(rows))          # register size: rows = cols = 2^m
N = rows * cols                 # search space: N coordinate pairs
M = len(void_coords)            # marked states: one per void
 
print(f"Grid        : {rows} x {cols}  (m={m})")
print(f"Search space: N = {N} coordinate pairs")
print(f"Qubits      : 2m = {n_qubits}  (vs {N} for flat encoding)")
print(f"Voids (M={M}): {void_coords}")
print()
print("Oracle expression:")
print(expression)

## Grover on microstructure

In [ ]:
n_qubits = int(np.log2(n_cells))       # qubits to address all cells
M = len(void_indices)                  # number of voids = marked states
print(f"Total cells: {n_cells}, Voids: {M}, Qubits needed: {n_qubits}")

qiskit_expr = ensure_variable_order(expression, n_qubits)
oracle = PhaseOracleGate(qiskit_expr, label="Void")
grover_op = grover_operator(oracle)

# Optimal K from Eq. (12.4)
theta = math.asin(math.sqrt(M / n_cells))
K = max(1, math.floor(math.pi / (4 * theta) - 0.5))
print(f"Grid: {int(np.sqrt(n_cells))}x{int(np.sqrt(n_cells))}, "
        f"N={n_cells} cells, M={M} void(s), K={K} iterations")
print(f"Void flat indices: {void_indices}")
print(f"Void bitstrings:   "
        f"{[format(k, f'0{n_qubits}b') for k in void_indices]}")

qc = QuantumCircuit(grover_op.num_qubits)
qc.h(range(n_qubits))
qc.compose(grover_op.power(K), inplace=True)
cr = ClassicalRegister(n_qubits)
qc.add_register(cr)
for i in range(n_qubits):
    qc.measure(i, cr[i])

shots = 1000
counts = simulate_measurements(qc, shots=shots)
print(counts)

## Parity oracle barebones

In [ ]:
# 4 Qubits: q0 is Ancilla, q1-q3 are Input
circuit = QuantumCircuit(4)

# 1. Prepare Ancilla (q0) in the |-> state
circuit.x(0)
circuit.h(0)
circuit.barrier()

# 2. Phase Oracle for |101> 
# Input register is (q3, q2, q1). For |101>, q2 must be flipped.
circuit.cx(1,0,ctrl_state='1')   # Flip q0 if q1 is 1
circuit.cx(2,0,ctrl_state='0')   # Flip q0 if q2 is 0
circuit.cx(3,0,ctrl_state='1')   # Flip q0 if q3 is 1
circuit.barrier()

# 3. Clean up Ancilla to return it to |0>
circuit.h(0)
circuit.x(0)

display(circuit.draw('mpl'))

# Verify the result
# The statevector will show a negative sign on the |1010> state 
# (where the last 0 is the ancilla q0)
psi = Statevector(circuit)
display(psi.draw('latex'))

## Parity Oracle Example

In [ ]:
# 4 Qubits: q0 is Ancilla, q1-q3 are Input
circuit = QuantumCircuit(4)

# 1. Prepare Ancilla (q0) in the |-> state
circuit.x(0)
circuit.h(0)
for qubit in [1, 2, 3]:
    circuit.h(qubit)
circuit.barrier()

# 2. Phase Oracle for |101> 
# Input register is (q3, q2, q1). For |101>, q2 must be flipped.
circuit.cx(1,0,ctrl_state='1')   # Flip q0 if q1 is 1
circuit.cx(2,0,ctrl_state='0')   # Flip q0 if q2 is 0
circuit.cx(3,0,ctrl_state='1')   # Flip q0 if q3 is 1
circuit.barrier()

# 3. Clean up Ancilla to return it to |0>
circuit.h(0)
circuit.x(0)

display(circuit.draw('mpl'))

# Verify the result
# The statevector will show a negative sign on the |1010> state 
# (where the last 0 is the ancilla q0)
psi = Statevector(circuit)
display(psi.draw('latex'))